In [1]:
import matplotlib.pyplot as plt #Supports our visualization
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import time


import statsmodels.api as sm
from scipy import stats
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier



df = pd.read_csv("letters.csv")

df.head()


,label,pixel43,pixel44,pixel92,pixel124,pixel125,pixel126,pixel127,pixel128,pixel129,...,pixel329,pixel351,pixel410,pixel411,pixel412,pixel413,pixel414,pixel415,pixel416,pixel417
0,1,0,0,0,0,0,0,0,0,0,...,0,254,0,0,0,0,0,0,0,0
1,0,0,0,0,137,137,192,86,72,1,...,254,0,0,75,254,254,254,17,0,0
2,1,0,0,0,3,141,139,3,0,0,...,0,184,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,94,255,69,0,0,0,0,0
4,0,0,0,0,155,254,254,254,157,30,...,253,0,0,0,223,253,253,253,129,0


Part 1 Data Cleansing and setting up

In [2]:
#Checks the shape and see if there are any missing or duplicate datas
print("Original shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Question Mark-missing data placeholders:", (df == "?").sum())


Original shape: (42000, 46)
Missing values: 0
Duplicate rows: 1633
Question Mark-missing data placeholders: label       0
pixel43     0
pixel44     0
pixel92     0
pixel124    0
pixel125    0
pixel126    0
pixel127    0
pixel128    0
pixel129    0
pixel130    0
pixel131    0
pixel132    0
pixel133    0
pixel134    0
pixel135    0
pixel136    0
pixel137    0
pixel138    0
pixel146    0
pixel147    0
pixel148    0
pixel149    0
pixel150    0
pixel151    0
pixel152    0
pixel153    0
pixel154    0
pixel155    0
pixel156    0
pixel157    0
pixel158    0
pixel159    0
pixel160    0
pixel327    0
pixel328    0
pixel329    0
pixel351    0
pixel410    0
pixel411    0
pixel412    0
pixel413    0
pixel414    0
pixel415    0
pixel416    0
pixel417    0
dtype: int64


In [3]:

#Deletes any duplicates from the datasets to prevent any negative influence from them
df = df.drop_duplicates().reset_index(drop=True)

x = df.drop(columns="label")

#The target variable that we are trying to classify from the predictors
y = df["label"]

#Split the datasets into  70% training and 30% test datasets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 42)

In [4]:
#This cell is used for standardscalar to make sure that there aren't any outliers in the datas

sc = StandardScaler()

x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)



Part 2: Building KNN model and evaulation

In [5]:
#This section uses GirdSearchCV to determine which tested value of K gives
#the highest average cross-validation accuracy for the KNN model

#List of K values that GridSearchCV will compare for the K-nearest neighbor algorithm
#it is short due to it crashing after running for so long but would still take a while
parameter_grid = { "n_neighbors": [4, 5, 7, 9, 11, 13, 15, 21]}

#Creates the initial KNN classifier 
knn = KNeighborsClassifier()

# Create the GridSearch object
grid_search = GridSearchCV(
    estimator=knn, #The model being evaluated
    param_grid = parameter_grid, #The K values tested
    cv=3, # uses 3-fold cross-validation
    scoring="accuracy"  # compare models using accuracy
)

#tests each value of K on the training data
grid_search.fit(x_train, y_train)

# displays the K value with highest cross validation accuracy and what it  acheived 
print("Best K:", grid_search.best_params_["n_neighbors"])
print(
    f"Best cross-validation accuracy: "
    f"{grid_search.best_score_:.2%}"
)

Best K: 15
Best cross-validation accuracy: 64.84%


In [6]:
# This cell would use the KNN mode with k value that gives the best accuracy on the 
#training data to be used on the test data sets 

#Retrieves the KNN model that had the highest cross-validation from GridSearchCV
best_knn_model = grid_search.best_estimator_

#Used the selected KNN model to predict number labels for data in the test set
predicted_labels = best_knn_model.predict(x_test)

# Reset the index of the actual test labels so that they
# line up with the positions of the predicted labels.
actual_labels = y_test.reset_index(drop=True)

# Display the first 15 test samples. The number could be replaced with len(n_test_samples
# but would produce a very long output that would take quite some time
for row in range(15):
    print(
        f"Test sample {row + 1}: "
        f"Predicted = {predicted_labels[row]}, " #Shows the predicted labels done by KNN
        f"Actual = {actual_labels.iloc[row]}, " # Shows the actual label to compare
        f"Match = {predicted_labels[row] == actual_labels.iloc[row]}" # Checks to see if they match or not
    )

# compares all predicted labels with all actual labels to calculate the final accuracy
# on the test set
test_accuracy = accuracy_score(
    actual_labels,
    predicted_labels
    
)
print("\nSelected K:", grid_search.best_params_["n_neighbors"]) # Shows what value K that was selected by GridSearchCV
print(f"Test accuracy: {test_accuracy:.2%}") #Shows the accuracy percentage


Test sample 1: Predicted = 2, Actual = 2, Match = True
Test sample 2: Predicted = 1, Actual = 3, Match = False
Test sample 3: Predicted = 1, Actual = 8, Match = False
Test sample 4: Predicted = 3, Actual = 3, Match = True
Test sample 5: Predicted = 5, Actual = 3, Match = False
Test sample 6: Predicted = 0, Actual = 0, Match = True
Test sample 7: Predicted = 1, Actual = 8, Match = False
Test sample 8: Predicted = 7, Actual = 7, Match = True
Test sample 9: Predicted = 6, Actual = 0, Match = False
Test sample 10: Predicted = 5, Actual = 4, Match = False
Test sample 11: Predicted = 7, Actual = 7, Match = True
Test sample 12: Predicted = 7, Actual = 7, Match = True
Test sample 13: Predicted = 8, Actual = 8, Match = True
Test sample 14: Predicted = 7, Actual = 7, Match = True
Test sample 15: Predicted = 8, Actual = 8, Match = True

Selected K: 15
Test accuracy: 65.54%


Part 3:Building Neural network and Evaluation

In [7]:
#This cell would go through each of the activation to see which would give the neural network the best accuracy

#The list of activation/link function used for neural network
activation_funcs = ["identity","logistic","tanh", "relu"]

#Empty list to hold results from the loop for activation_func
activation_results = []

#Goes through each of the activation functions to see how each performs
for activation in activation_funcs:

    #Makes the neural network with the given stats and has the activation included in it
    mlp_model = MLPClassifier(hidden_layer_sizes = (60,), # Creates one hidden layer containing 60 neurons
                              activation = activation, #uses a function from the list
                              solver = 'sgd', # Uses stochastic gradient descent to update the model's weights
                              learning_rate_init = 0.05,  # Controls how much the weights change during each update
                              max_iter = 15000, # Sets the maximum number of training iterations
                              random_state = 42 # Makes the neural-network results reproducible
                             )

    #Trains the model
    mlp_model.fit(x_train_scaled,y_train)

    #Get the accuracy for both training and test datasets
    training_accuracy = mlp_model.score(x_train_scaled,y_train)
    testing_accuracy = mlp_model.score(x_test_scaled,y_test)

    #Saves their results into the list
    activation_results.append({"Activation": activation,
        "Training Accuracy": training_accuracy,
        "Testing Accuracy": testing_accuracy
    })

#Turns the list into a data frame
activation_results_df = pd.DataFrame(activation_results)


#Sorts the activation func based on their testing accuracy from highest to lowest
activation_results_df = activation_results_df.sort_values(
    by="Testing Accuracy",
    ascending=False
).reset_index(drop=True)

activation_results_df

,Activation,Training Accuracy,Testing Accuracy
0,relu,0.722148,0.684584
1,logistic,0.736410,0.682685
2,tanh,0.737755,0.680951
3,identity,0.590140,0.590703


Part 4: Benchmarking between KNN and neural network

In [8]:
best_k = grid_search.best_params_["n_neighbors"] #Gets the k value that gives best accuracy

# Dictionary containing the final KNN and neural-network models
# that will be compared using the same training and testing data
benchmark_models= {
    "KNN": KNeighborsClassifier(
        n_neighbors=best_k),
    
    "Neural Network": MLPClassifier(
        hidden_layer_sizes=(60,),  # Creates one hidden layer containing 60 neurons
        activation="relu", #
        solver="sgd",
        learning_rate_init=0.05,
        max_iter=15000,
        random_state=42
    )
}

# Empty list used to store the benchmark results for each model
benchmark_results = []

# Goes through each model in the benchmark_models dictionary
for model_name, model in benchmark_models.items():
    
    # Records the time immediately before training begins
    start_time = time.perf_counter()

    model.fit(x_train_scaled, y_train) # Trains the model using the scaled training predictors

    # Calculates the model's accuracy on the test dataset
    # For classifiers, .score() returns classification accuracy
    testing_accuracy = model.score(
        x_test_scaled,
        y_test
    )

    # Calculates the time required to train the model and score
    # the test dataset
    computation_time = (
        time.perf_counter() - start_time 
    )

    # Calculates the model's accuracy on the training dataset
    training_accuracy = model.score(
        x_train_scaled,
        y_train
    )

    # Calculates the difference between training and testing accuracy
    # A smaller gap indicates a more consistent model fit
    accuracy_gap = abs(
        training_accuracy - testing_accuracy
    )

    # Stores the accuracy, fit, and computation-time results
    # for the current model
    benchmark_results.append({
        "Model": model_name,
        "Training Accuracy": training_accuracy,
        "Testing Accuracy": testing_accuracy,
        "Accuracy Gap": accuracy_gap,
        "Computation Time (Seconds)": computation_time
    })

# Converts the stored benchmarking results into a DataFrame
# so the two models can be compared in a table
benchmark_results_df = pd.DataFrame(
    benchmark_results
)
# Displays the final benchmark comparison
benchmark_results_df


,Model,Training Accuracy,Testing Accuracy,Accuracy Gap,Computation Time (Seconds)
0,KNN,0.687394,0.647098,0.040296,0.524508
1,Neural Network,0.722148,0.684584,0.037563,11.271467
